In [ ]:
import csv, hashlib, os
import pandas as pd

RAW_FILE = "netflix_titles.csv"          # name of the raw file
OUT_DIR = "output"
os.makedirs(OUT_DIR, exist_ok=True)

# Switches for the "nice to have" column of the task table (the minimum steps always run)
NICE_STRINGS = False      # curly quotes -> straight, collapse double spaces, remove odd unicode characters
NICE_FUZZY_DUPS = False   # ignore case/punctuation when comparing titles (removes more rows - agree with the team first)

EXPECTED_COLUMNS = ["show_id", "type", "title", "director", "cast", "country",
                    "date_added", "release_year", "rating", "duration",
                    "listed_in", "description"]

# Valid certificates (edit if your team's list differs)
VALID_RATINGS = {"G", "PG", "PG-13", "R", "NC-17", "NR", "UR",
                 "TV-Y", "TV-Y7", "TV-Y7-FV", "TV-G", "TV-PG", "TV-14", "TV-MA"}

In [ ]:
#upload the raw file
if not os.path.exists(RAW_FILE):
    try:
        from google.colab import files            # only exists inside Colab
        uploaded = files.upload()
        RAW_FILE = list(uploaded.keys())[0]        # Colab may rename it, e.g. "netflix_titles (1).csv"
    except ImportError:
        raise FileNotFoundError(f"{RAW_FILE} not found - put it next to the notebook.")

#read everything as text
df = pd.read_csv(RAW_FILE, dtype=str, keep_default_na=False, na_values=[], encoding="utf-8")

#all 12 expected columns must exist
missing_cols = [c for c in EXPECTED_COLUMNS if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing expected column(s): {missing_cols}")
extra_cols = [c for c in df.columns if c not in EXPECTED_COLUMNS]
if extra_cols:
    raise ValueError(f"Unexpected extra column(s): {extra_cols}")
df = df[EXPECTED_COLUMNS]

#row count must match the file (count CSV *records*, not text lines)
with open(RAW_FILE, encoding="utf-8", newline="") as f:
    file_records = sum(1 for _ in csv.reader(f)) - 1          #minus header
assert len(df) == file_records, f"row mismatch: pandas {len(df)} vs file {file_records}"

#log file size, checksum, rows and columns (nice to have)
print("file     :", RAW_FILE)
print("size     :", f"{os.path.getsize(RAW_FILE):,} bytes")
print("sha256   :", hashlib.sha256(open(RAW_FILE, "rb").read()).hexdigest())
print("rows x cols:", df.shape[0], "x", df.shape[1])
rows_raw = len(df)
df.head(3)

Saving netflix_titles.csv to netflix_titles.csv
file     : netflix_titles.csv
size     : 3,399,671 bytes
sha256   : df1f4ad2027a5a14c3a33932ef0d4054565ff88adf92b7f263601b70fdc6f3f3
rows x cols: 8807 x 12


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...


In [ ]:
before_ws = {c: int((df[c] != df[c].str.strip()).sum()) for c in df.columns}
print("cells with leading/trailing whitespace BEFORE:", {k: v for k, v in before_ws.items() if v})

for c in df.columns:
    df[c] = df[c].str.strip()                 # also removes non-breaking spaces

if NICE_STRINGS:                              # nice to have (This dictionary maps special Unicode quotation marks to standard quotation marks)
    quote_map = {"\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"'}
    for c in df.columns:
        for k, v in quote_map.items():
            df[c] = df[c].str.replace(k, v, regex=False)
        df[c] = (df[c].str.replace("\u00a0", " ", regex=False)
                      .str.replace("[\u200b\u200c\u200d\ufeff]", "", regex=True)
                      .str.replace(r"\s{2,}", " ", regex=True)
                      .str.strip())

df = df.replace("", pd.NA)                    # empty string -> missing

print("\nmissing values per column AFTER:")
print(df.isna().sum().to_string())
print("\ndate_added sample:", df["date_added"].dropna().head(3).tolist())
df.head()

cells with leading/trailing whitespace BEFORE: {'title': 1, 'date_added': 88}

missing values per column AFTER:
show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0

date_added sample: ['September 25, 2021', 'September 24, 2021', 'September 24, 2021']


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,<NA>,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,<NA>,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",<NA>,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,<NA>,<NA>,<NA>,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,<NA>,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [ ]:
# Identify rows where rating has the minutes but duration is missing or empty
looks_like_minutes = df["rating"].str.match(r"^\d+\s*min$", na=False)
is_duration_empty = df["duration"].isna() | (df["duration"] == "")
shifted = looks_like_minutes & is_duration_empty

print("rows to fix:", int(shifted.sum()))
display(df.loc[shifted, ["show_id", "title", "rating", "duration"]])

conflict = looks_like_minutes & df["duration"].notna() & (df["duration"] != "")
if conflict.any():
    print("WARNING - minutes in rating AND a duration present, check manually:")
    display(df.loc[conflict, ["show_id", "title", "rating", "duration"]])

df.loc[shifted, "duration"] = df.loc[shifted, "rating"]
df.loc[shifted, "rating"] = pd.NA

print("\nafter the fix:")
display(df.loc[shifted, ["show_id", "title", "rating", "duration"]])

# quarantine ratings that are not valid certificates
bad_rating = df["rating"].notna() & ~df["rating"].isin(VALID_RATINGS)
df[bad_rating].to_csv(f"{OUT_DIR}/quarantine_ratings.csv", index=False)
print("ratings not in the valid list (-> quarantine_ratings.csv):", int(bad_rating.sum()))

rows to fix: 3


,show_id,title,rating,duration
5541,s5542,Louis C.K. 2017,74 min,<NA>
5794,s5795,Louis C.K.: Hilarious,84 min,<NA>
5813,s5814,Louis C.K.: Live at the Comedy Store,66 min,<NA>



after the fix:


,show_id,title,rating,duration
5541,s5542,Louis C.K. 2017,<NA>,74 min
5794,s5795,Louis C.K.: Hilarious,<NA>,84 min
5813,s5814,Louis C.K.: Live at the Comedy Store,<NA>,66 min


ratings not in the valid list (-> quarantine_ratings.csv): 0


In [ ]:
#exact duplicate rows
exact = df.duplicated(keep="first")
removed = [df[exact].assign(removed_because="exact duplicate row")]
df = df[~exact]

#(same title + type + release_year), we will keep first
if NICE_FUZZY_DUPS:
    norm_title = (df["title"].str.lower()
                  .str.replace(r"[^\w\s]", "", regex=True)
                  .str.replace(r"\s+", " ", regex=True).str.strip())
    same_key = pd.concat([norm_title, df["type"], df["release_year"]], axis=1).duplicated(keep="first")
else:
    same_key = df.duplicated(subset=["title", "type", "release_year"], keep="first")
removed.append(df[same_key].assign(removed_because="same title + type + release_year"))
df = df[~same_key]

removed_df = pd.concat(removed)
removed_df.to_csv(f"{OUT_DIR}/removed_duplicates.csv", index=False)
print("rows removed:", len(removed_df))
removed_df[["show_id", "title", "type", "release_year", "removed_because"]]

rows removed: 1


,show_id,title,type,release_year,removed_because
6529,s6530,Consequences,Movie,2014,same title + type + release_year


In [ ]:
# Step 1: columns and row count
assert list(df.columns) == EXPECTED_COLUMNS, "column list/order changed"

# Step 2: no untrimmed text, no empty strings, empty director became missing
def has_edge_space(s):
    s = s.dropna()
    return bool((s.str.startswith((" ", "\u00a0")) | s.str.endswith((" ", "\u00a0"))).any())
assert not any(has_edge_space(df[c]) for c in df.columns), "untrimmed text left"
assert not (df.fillna("x") == "").any().any(), "empty strings still present"
assert df["director"].isna().sum() > 0
assert df["date_added"].dropna().str.match(r"^[A-Z]").all(), "date_added still has a leading space"

# Step 3: '74 min' moved to duration, normal rows untouched
louis = df[df["title"] == "Louis C.K. 2017"].iloc[0].to_dict()
assert louis["duration"] == "74 min", "74 min not moved to duration"
assert pd.isna(louis["rating"]), "rating not set to NA"
assert not df["rating"].dropna().str.contains("min").any(), "'min' values still in rating"

# Step 4: duplicate gone, different-year remakes kept
assert not df.duplicated(subset=["title", "type", "release_year"]).any()
assert (df["title"] == "Consequences").sum() == 1, "duplicated title still present"
assert (df["title"].str.lower() == "fullmetal alchemist").sum() == 2, "different-year remakes must be kept"

print("ALL CHECKS PASSED")
print(f"rows: {rows_raw} raw -> {len(df)} clean ({rows_raw - len(df)} removed)")

ALL CHECKS PASSED
rows: 8807 raw -> 8806 clean (1 removed)


In [ ]:
df.to_csv(f"{OUT_DIR}/netflix_clean.csv", index=False, encoding="utf-8")   # index=False: no extra column
print(os.listdir(OUT_DIR))

# In Colab: download the three files to your computer (then upload them to GitHub)
try:
    from google.colab import files
    for name in ["netflix_clean.csv", "removed_duplicates.csv", "quarantine_ratings.csv"]:
        files.download(f"{OUT_DIR}/{name}")
except ImportError:
    print("Not running in Colab - files are in the 'output' folder.")

['removed_duplicates.csv', 'quarantine_ratings.csv', 'netflix_clean.csv']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>